# Ejercicio: Web Scraping

## Nombre: Edison Quizhpe

## Objetivo de la práctica

El objetivo de este ejercicio es construir un web scraper que recoja datos de un website.

rag_corpus
1. Identificar los datos que quieres obtener.
2. Elegir el sitio web objetivo.
3. Planificar la estructura del corpus.

## Parte 1: Entender el sitio web objetivo

- Analizar la estructura de la página web a ser analizada.
- Identificar los elementos HTML que contienen los datos bsuscados.

In [3]:
from bs4 import BeautifulSoup

file = 'rotisserie-chicken.html'

# Load the HTML file
with open(file, "r", encoding="utf-8") as file:
    html_content = file.read()
    
# Parse the HTML content with BeautifulSoup
soup = BeautifulSoup(html_content, "html.parser")

In [4]:
# Extracting the recipe title
title = soup.find("meta", {"property": "og:title"})["content"]
title

'Rotisserie Chicken'

In [5]:
ingredients_section = soup.find_all("li", class_="mm-recipes-structured-ingredients__list-item")
for ingredient in ingredients_section:
    print(ingredient.text.strip())

1 (3 pound) whole chicken
1 pinch salt
¼ cup butter, melted
1 tablespoon salt
1 tablespoon ground paprika
¼ tablespoon ground black pepper


## Parte 2: Obtener los datos deseados

* Buscar dentro del contenido HTML y extraer la información.

In [6]:
# Extracting the description
description = soup.find("meta", {"name": "description"})["content"]

# Extracting the ingredients
ingredients_section = soup.find_all("li", class_="mm-recipes-structured-ingredients__list-item")
ingredients = [ingredient.get_text().strip() for ingredient in ingredients_section]

# Extracting the instructions
instructions_section = soup.find_all("p", class_="comp mntl-sc-block mntl-sc-block-html")
instructions = [instruction.get_text().strip() for instruction in instructions_section]

# Extracting the nutrition information
nutrition_section = soup.find_all("span", class_="mm-recipes-nutrition-facts-label__nutrient-name mm-recipes-nutrition-facts-label__nutrient-name--has-postfix")
nutrition_facts = [fact.parent.get_text().strip().replace('\n', ' ') for fact in nutrition_section]

# Print the extracted information
print("Recipe Title:", title)
print("Description:", description)
print("Ingredients:")
for ingredient in ingredients:
    print("-", ingredient)
print("Instructions:")
for i, instruction in enumerate(instructions, 1):
    print(f"{i}. {instruction}")
print("Nutrition Facts:")
for fact in nutrition_facts:
    print("-", fact)


Recipe Title: Rotisserie Chicken
Description: Rotisserie chicken that's easy to cook on a gas grill and turns out moist and juicy with crispy skin. This is a simple recipe that our family loves.
Ingredients:
- 1 (3 pound) whole chicken
- 1 pinch salt
- ¼ cup butter, melted
- 1 tablespoon salt
- 1 tablespoon ground paprika
- ¼ tablespoon ground black pepper
Instructions:
1. Intimidated by the idea of making a rotisserie chicken at home? We're here to help. Get your grill and rotisserie attachment ready — you'll want to try this recipe ASAP.
2. Here's what you'll need to make rotisserie chicken at home:
3. · Whole Chicken: This recipe is meant for a whole 3-pound chicken. If your chicken is larger or smaller, you'll have to adjust the cooking time.· Butter: Butter keeps the chicken moist and juicy, while giving the seasonings something to stick to.· Seasonings: The rotisserie chicken is simply seasoned with salt, pepper, and paprika.
4. You'll find the full, step-by-step recipe below — b

## Parte 3: Obtener enlaces relacionados
* Encontrar links a otras recetas para completar el corpus

In [7]:
# Find all the links to other recipes
recipe_links = soup.find_all("a", href=True)

# Filter and print only the links that are likely to be recipes
recipe_urls = []
for link in recipe_links:
    href = link['href']
    if "recipe" in href:
        recipe_urls.append(href)

# Print the recipe URLs
print("Linked Recipes:")
for url in recipe_urls:
    print(url)

Linked Recipes:
https://www.allrecipes.com/authentication/login?regSource=3675&relativeRedirectUrl=%2Frecipe%2F93168%2Frotisserie-chicken%2F
/account/add-recipe
https://www.myrecipes.com/favorites
https://www.allrecipes.com/authentication/logout?relativeRedirectUrl=%2Frecipe%2F93168%2Frotisserie-chicken%2F
https://www.magazines.com/allrecipes-magazine.html?utm_source=allrecipes.com&utm_medium=owned&utm_campaign=i111arr1w2661
https://www.magazines.com/allrecipes-magazine.html
https://www.allrecipes.com/recipes/17562/dinner/
https://www.allrecipes.com/recipes/17057/everyday-cooking/more-meal-ideas/5-ingredients/main-dishes/
https://www.allrecipes.com/recipes/15436/everyday-cooking/one-pot-meals/
https://www.allrecipes.com/recipes/1947/everyday-cooking/quick-and-easy/
https://www.allrecipes.com/recipes/455/everyday-cooking/more-meal-ideas/30-minute-meals/
https://www.allrecipes.com/recipes/17889/everyday-cooking/family-friendly/family-dinners/
https://www.allrecipes.com/recipes/94/soups-s

## Parte 4: Hacer RAG con las recetas obtenidas
* Una vez que se ha construido el corpus, implementar y desplegar RAG para realizar búsquedas en el corpus

In [54]:
from pathlib import Path
from urllib.parse import urljoin, urlparse
from collections import deque
import re, requests
from bs4 import BeautifulSoup

base_url = "https://www.allrecipes.com"
recipe_pattern = re.compile(r"^/recipe/\d+/")

# 1. Configuración de directorios simplificada
notebook_dir = next((d for d in [Path.cwd(), Path.cwd() / "12webcrawling"] if (d / "rotisserie-chicken.html").exists()), None)
if not notebook_dir:
    raise FileNotFoundError("No se encontró rotisserie-chicken.html en la carpeta actual ni en 12webcrawling/")

html_cache_dir = notebook_dir / "rag_corpus" / "html"
html_cache_dir.mkdir(parents=True, exist_ok=True)

session = requests.Session()
session.headers.update({"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/126.0"})

max_depth = 2
max_documents = 200

# 2. Normalización compacta
def normalize_url(href, current_url):
    if not href: return None
    url = urlparse(urljoin(current_url or base_url, href).split("#")[0].rstrip("/"))
    if (url.scheme in ("http", "https") and "allrecipes.com" in url.netloc and 
        recipe_pattern.match(url.path) and not any(b in url.path for b in ("login", "logout", "authentication"))):
        return url.geturl()
    return None

# 3. Extracción de datos reducida
def extract_recipe(soup, url):
    get_meta = lambda prop, attr: (soup.find("meta", {prop: attr}) or {}).get("content", "").strip()
    get_texts = lambda tag, cls: [e.get_text(" ", strip=True) for e in soup.find_all(tag, class_=re.compile(cls))]

    data = {
        "source_url": url,
        "title": get_meta("property", "og:title") or url,
        "description": get_meta("name", "description"),
        "ingredients": get_texts("li", "mm-recipes-structured-ingredients__list-item"),
        "instructions": get_texts("p", "mntl-sc-block-html"),
        "nutrition": [f.parent.get_text(" ", strip=True) for f in soup.find_all("span", class_=re.compile("nutrient-name--has-postfix"))]
    }

    # Construcción del texto
    parts = [data["title"]] + ([f"Descripción: {data['description']}"] if data["description"] else [])
    if data["ingredients"]: parts.append("Ingredientes:\n" + "\n".join(f"- {i}" for i in data["ingredients"]))
    if data["instructions"]: parts.append("Instrucciones:\n" + "\n".join(f"{i}. {paso}" for i, paso in enumerate(data["instructions"], 1)))
    if data["nutrition"]: parts.append("Información nutricional:\n" + "\n".join(f"- {n}" for n in data["nutrition"]))
    
    data["text"] = "\n\n".join(parts)
    return data

# 4. Bucle principal del Crawler
seed_path = notebook_dir / "rotisserie-chicken.html"
local_html_files = sorted(
    path for path in notebook_dir.rglob("*.html")
    if "rag_corpus" not in path.parts
)
seed_paths = local_html_files or [seed_path]

queue = deque((path.resolve().as_uri(), path, 0) for path in seed_paths)
queued_urls = {path.resolve().as_uri() for path in seed_paths}
visited, corpus = set(), []

while queue and len(corpus) < max_documents:
    url, path, depth = queue.popleft()
    if url in visited: continue
    visited.add(url)

    try:
        # Lógica de caché en línea
        if path.exists():
            html = path.read_text(encoding="utf-8")
        else:
            resp = session.get(url, timeout=20)
            resp.raise_for_status()
            html = resp.text
            path.write_text(html, encoding="utf-8")
    except requests.RequestException as exc:
        print(f"Error descargando {url}: {exc}")
        continue

    soup = BeautifulSoup(html, "html.parser")
    record = extract_recipe(soup, url)
    
    # Extraer enlaces válidos en una línea usando filter y comprensión
    related_links = list(dict.fromkeys(filter(None, (normalize_url(a.get("href"), url) for a in soup.find_all("a")))))
    record.update({"related_links": related_links, "crawl_depth": depth, "local_html_path": str(path.resolve())})
    corpus.append(record)

    if depth < max_depth:
        for link in related_links:
            if link not in visited and link not in queued_urls:
                slug = Path(urlparse(link).path.rstrip("/")).name or "recipe"
                queue.append((link, html_cache_dir / f"{slug}.html", depth + 1))
                queued_urls.add(link)

print(f"Carpeta del notebook: {notebook_dir.resolve()}\nArchivos HTML locales encontrados: {len(local_html_files)}\nProfundidad máxima: {max_depth}\nLímite de documentos: {max_documents}\nRecetas descargadas: {len(corpus)}")

Carpeta del notebook: C:\Users\wwwed\OneDrive - Escuela Politécnica Nacional\Escritorio\EPN\SEPTIMO SEMESTRE\ir26a\12webcrawling
Archivos HTML locales encontrados: 1
Profundidad máxima: 2
Límite de documentos: 200
Recetas descargadas: 147


In [50]:
corpus_df = pd.DataFrame(corpus)
corpus_jsonl_path = corpus_dir / "recipes_corpus.jsonl"
corpus_csv_path = corpus_dir / "recipes_corpus.csv"

with corpus_jsonl_path.open("w", encoding="utf-8") as jsonl_file:
    for record in corpus:
        jsonl_file.write(json.dumps(record, ensure_ascii=False) + "\n")

corpus_df.to_csv(corpus_csv_path, index=False, encoding="utf-8")

In [59]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import os
from pathlib import Path
from dotenv import load_dotenv, dotenv_values
from IPython.display import display, Markdown

if "corpus_df" not in globals():
    raise NameError("Primero ejecuta la celda que carga corpus_df desde rag_corpus.")

env_candidates = []
if "notebook_dir" in globals():
    env_candidates.append(Path(notebook_dir) / ".env")
env_candidates.append(Path.cwd() / ".env")
env_candidates.append(Path.cwd() / "12webcrawling" / ".env")

env_path = None
for candidate in env_candidates:
    if candidate.exists():
        env_path = candidate
        load_dotenv(candidate)
        break

corpus_df = corpus_df.copy()
corpus_df["rag_text"] = corpus_df.apply(
    lambda row: "\n".join(
        part for part in [
            f"Título: {row.get('title', '')}",
            f"Descripción: {row.get('description', '')}",
            f"Ingredientes: {row.get('ingredients', '')}",
            f"Instrucciones: {row.get('instructions', '')}",
            f"Nutrición: {row.get('nutrition_facts', '')}",
        ]
        if str(part).strip() and str(part).strip() not in {"Título:", "Descripción:", "Ingredientes:", "Instrucciones:", "Nutrición:"}
    ),
    axis=1,
)

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
doc_embeddings = model.encode(
    corpus_df["rag_text"].tolist(),
    normalize_embeddings=True,
    show_progress_bar=False,
)


def rag_search(query, top_k=3):
    query_embedding = model.encode([query], normalize_embeddings=True)
    scores = cosine_similarity(query_embedding, doc_embeddings)[0]
    ranked_indices = scores.argsort()[::-1][:top_k]
    results = []
    for rank, idx in enumerate(ranked_indices, 1):
        row = corpus_df.iloc[idx]
        results.append(
            {
                "rank": rank,
                "title": row.get("title", ""),
                "source_url": row.get("source_url", ""),
                "score": float(scores[idx]),
                "description": row.get("description", ""),
                "ingredients": row.get("ingredients", ""),
                "instructions": row.get("instructions", ""),
                "related_links": row.get("related_links", ""),
            }
        )
    return results


def build_gemini_context(results):
    context_lines = []
    for item in results:
        context_lines.append(
            f"[{item['rank']}] {item['title']} | similitud={item['score']:.3f}\n"
            f"URL: {item['source_url']}\n"
            f"Descripción: {item['description']}\n"
            f"Ingredientes: {item['ingredients']}\n"
            f"Instrucciones: {item['instructions']}\n"
        )
    return "\n".join(context_lines)

query = "beer"
retrieved_results = rag_search(query, top_k=10)
context = build_gemini_context(retrieved_results)

display(Markdown(context))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6875.47it/s]


[1] Beer Floats | similitud=0.432
URL: https://www.allrecipes.com/recipe/270577/beer-floats
Descripción: Beer floats are the next best drink you've never had. Play around with Chef John's two winning combos: stout with coffee ice cream and raspberry sour with vanilla.
Ingredientes: ['1 cup chocolate stout beer', '1 scoop coffee ice cream', '1 pinch unsweetened cocoa powder', '1 cup Belgian-style raspberry sour beer', '1 scoop vanilla ice cream']
Instrucciones: ['Version 1: Pour stout into a British pint glass and scoop coffee ice cream on top. Garnish with a dusting of cocoa powder.', 'Version 2: Pour raspberry beer into a pint glass; top with vanilla ice cream.', 'Add a straw to each float and serve with spoons alongside.']

[2] Stout and Ale | similitud=0.430
URL: https://www.allrecipes.com/recipe/216528/stout-and-ale
Descripción: This black and tan recipe is called a "half and half" in Ireland. A pint glass is filled halfway with lager on the bottom with a float of Guinness on top.
Ingredientes: ['1 (12 fluid ounce) bottle lager beer (such as Harp®)', '1 (12 fluid ounce) can or bottle Irish stout beer (such as Guinness®)']
Instrucciones: ['Divide lager beer evenly between 2 tall beer glasses.', 'Working with one glass at a time, place a large tablespoon, dome-side up, 1 inch or so above lager beer, with the tip of the spoon pointed slightly downward.', 'Slowly pour 1/2 of the stout beer over the tablespoon so it gently pours down the side of the glass in a thick trickle. Allow to stand until 2 distinct layers of beer form, 3 to 5 seconds. Repeat with the remaining glass of beer.']

[3] Irish Slammer | similitud=0.373
URL: https://www.allrecipes.com/recipe/22721/irish-slammer
Descripción: An Irish slammer is similar to a boilermaker. Drop a shot glass full of Irish whiskey and Irish cream into a glass of stout beer, and drink it all at once!
Ingredientes: ['6 fluid ounces Irish stout beer', '¾ fluid ounce Irish whiskey', '¾ fluid ounce Irish cream liqueur']
Instrucciones: ['Pour beer into a pint glass.', 'Pour whiskey and cream liqueur into a shot glass. Drop the shot glass into the pint glass, and drink the entire contents at once.']

[4] Vermont Maple Stout Baked Beans | similitud=0.357
URL: https://www.allrecipes.com/recipe/276708/vermont-maple-stout-baked-beans
Descripción: Vermont maple syrup is combined with a Vermont maple breakfast stout, salt pork, and dried beans resulting in a sweet and savory batch of slow-cooked baked beans.
Ingredientes: ['½ pound dried pinto beans', '½ pound dried navy beans', '½ teaspoon baking soda', 'water to cover', '12 ounces salt pork, rind removed', '1 sweet white onion, cut into thick slices', '1 (12 fluid ounce) can or bottle stout beer (such as 14th Star Brewing Company Maple Breakfast Stout)', '1 ½ cups hot water', '¼ cup Vermont maple syrup', '¼ cup brown sugar', '3 tablespoons molasses', '2 tablespoons balsamic vinegar', '1 teaspoon Worcestershire sauce', '1 teaspoon ground paprika', '1 teaspoon garlic salt', '1 teaspoon freshly ground black pepper', '1 teaspoon mustard powder', '¾ teaspoon liquid smoke', 'salt and ground black pepper to taste']
Instrucciones: ['Combine pinto beans, navy beans, and baking soda in a large pot. Add water until level is 2 inches higher than beans. Heat over medium-high heat until boiling. Cook for 5 minutes then remove pot from heat. Set aside to soak for 8 hours.', 'Drain beans, reserving 1 1/2 cups soaking liquid. Place beans in a slow cooker.', 'Place salt pork in a large skillet over high heat; sear until browned, 3 to 5 minutes per side. Remove from the heat, cut into small chunks, and add to the slow cooker.', 'Add onion to the skillet with the leftover pork fat; cook over medium-high heat until translucent, about 5 minutes; transfer to the slow cooker.', 'Combine stout beer, hot water, maple syrup, brown sugar, molasses, balsamic vinegar, Worcestershire sauce, paprika, garlic salt, pepper, mustard powder, and liquid smoke in a bowl. Mix well and pour sauce over beans, pork, and onions in the slow cooker. Add the reserved bean soaking liquid.', 'Cook on Low until beans are tender, about 10 hours. Drain to desired consistency before serving.']

[5] The Best Beer Can Chicken Ever | similitud=0.330
URL: https://www.allrecipes.com/recipe/228070/the-best-beer-can-chicken-ever
Descripción: A whole grilled chicken gets a burst of flavor from a warm and complex spice rub and a can of dark stout beer with hot peppers and garlic cloves.
Ingredientes: ['1 cup chocolate stout beer', '3 green Thai chile peppers', '3 cloves garlic, peeled', '3 tablespoons brown sugar', '2 teaspoons dry mustard', '1 teaspoon garam masala', '1 teaspoon kosher salt', '1 teaspoon ground black pepper', '½ teaspoon cayenne pepper', '½ teaspoon ground cumin', '½ teaspoon ground cinnamon', '½ teaspoon onion powder', '½ teaspoon garlic powder', '¼ teaspoon ground nutmeg', '1 (5 pound) whole chicken']
Instrucciones: ['Preheat grill for medium heat. If using charcoal, push coals to the side of the grilling area for indirect heat.', 'Pour stout beer into an empty 12-ounce soda can and drop Thai chilies and garlic cloves into the can. Mix brown sugar, dry mustard, garam masala, kosher salt, black pepper, cayenne pepper, cumin, cinnamon, onion powder, garlic powder, and nutmeg in a bowl.', 'Rinse the chicken and coat the skin with the entire batch of spice rub. Place soda can containing beer mixture onto the prepared grill and sit the chicken upright onto the can.', 'Grill chicken over indirect heat until juices run clear and an instant-read meat thermometer inserted into the thickest part of the breast, not touching bone, reads at least 165 degrees F (75 degrees C), about 45 minutes. Let chicken rest 10 minutes before serving.']

[6] Chicken With Stout | similitud=0.301
URL: https://www.allrecipes.com/recipe/231695/chicken-with-stout
Descripción: Chicken thighs are pan-fried until golden brown and simmered with potatoes, carrots, and cabbage in dark Irish beer for a one-dish meal that makes a nice supper for St. Patrick's Day.
Ingredientes: ['½ cup all-purpose flour', '1 teaspoon salt, or to taste', '1 teaspoon ground black pepper, or to taste', '4 chicken thighs', '1 tablespoon vegetable oil', '4 potatoes, cut into 1-inch pieces', '2 large carrots, sliced', '½ head green cabbage, sliced', '1 ½ cups warm beef stock', '1 ½ cups Irish stout beer (such as Guinness®)', '1 tablespoon brown sugar', '1 tablespoon cornstarch', '2 tablespoons water']
Instrucciones: ['Preheat oven to 325 degrees F (165 degrees C).', 'Whisk flour, salt, and black pepper in a bowl. Press chicken thighs into seasoned flour to coat. Heat vegetable oil in a skillet over medium heat; fry coated chicken in the hot oil until golden brown, 5 to 8 minutes per side.', 'Place potatoes, carrots, and cabbage into a large casserole dish and lay chicken thighs on top of vegetables. Stir beef stock, stout beer, and brown sugar in a bowl. Mix cornstarch and water in a separate small bowl; stir into beer mixture. Pour beer mixture over chicken and vegetables. Cover the casserole dish.', 'Bake in the preheated oven for 1 hour and 15 minutes; uncover casserole and continue baking until chicken is cooked all the way through and sauce has thickened, 45 more minutes.']

[7] Clay's Grilled Beer Can Chicken | similitud=0.288
URL: https://www.allrecipes.com/recipe/228068/clays-grilled-beer-can-chicken
Descripción: A slightly smoky spice rub and a can of beer transform basic chicken into Clay's grilled beer can chicken. This recipe is the best you will ever make.
Ingredientes: ['1 tablespoon onion powder', '1 tablespoon salt', '1 tablespoon smoked paprika', '1 ½ teaspoons ground cumin', '1 ½ teaspoons ground cayenne pepper', '1 teaspoon garlic powder', '1 teaspoon dried oregano', '1 teaspoon dried thyme', '1 teaspoon brown sugar', '1 (4 pound) whole chicken, rinsed and dried', '1 tablespoon vegetable oil, or more as needed', '½ (12 ounce) can beer (such as Budweiser)']
Instrucciones: ['Whisk onion powder, salt, smoked paprika, cumin, cayenne pepper, garlic powder, oregano, thyme, and brown sugar together in a small bowl until thoroughly combined.', 'Make 2 cuts in chicken skin, 1 on front and 1 on back; loosen skin without tearing by inserting your hand underneath skin. Rub chicken skin and cavity with vegetable oil; rub spice rub over chicken skin, inside cavity, and beneath loosened skin.', 'Preheat an outdoor grill for medium heat.', 'Place the beer can on the preheated grill; slide chicken onto beer can by the cavity. Grill until chicken is thoroughly browned, the juices run clear, and an instant-read thermometer inserted into the breast, near but not touching bone, reads at least 165 degrees F (75 degrees C), 1 to 1 ½ hours.']

[8] Drunk Chicken | similitud=0.278
URL: https://www.allrecipes.com/recipe/19944/drunk-chicken
Descripción: Drunken chicken is cooked over the grill with a beer can up inside for a fun and easy dish!
Ingredientes: ['1 (2 to 3 pound) whole chicken', '1 (12 fluid ounce) can beer', '5 tablespoons poultry seasoning', '4 dashes liquid smoke flavoring', '4 bay leaves', '1 long metal skewer']
Instrucciones: ['Rinse and dry chicken. Remove excess fat and leave skin on. Lift skin from breast and thigh areas; slide bay leaves under skin. Coat chicken with poultry seasoning.', 'Drink 1/2 of the can of beer; pour liquid smoke into remaining beer. Raise tab on beer can until it is in the straight-up position.', 'Insert beer can into chicken from the bottom until even with bottom of chicken. Insert skewer through wing, ribs, beer can tab, and out the opposite side. (This keeps the can from falling out of chicken.)', 'Preheat the grill by lighting the coals. Spread the coals to form a ring around the outside edge of the grill.', 'Place chicken in the center, standing up on the can to cook. Close grill; cook for 2 hours. A meat thermometer inserted into the center should read at least 165 degrees F (74 degrees C).', 'Remove chicken carefully from the grill so as not to spill the contents of the can. Remove skewer and beer can; let chicken sit for 15 minutes before cutting.']

[9] Beer Can Chicken | similitud=0.272
URL: https://www.allrecipes.com/recipe/214618/beer-can-chicken
Descripción: This beer can chicken cooks a perfectly seasoned whole chicken on a grill until it's deliciously crisp and moist, full of flavor, and easy to carve.
Ingredientes: ['⅓ cup brown sugar', '2 tablespoons chili powder', '2 tablespoons paprika', '2 teaspoons dry mustard', '½ teaspoon salt', '¼ teaspoon ground black pepper', '½ (12 fluid ounce) can beer', '1 (3 pound) whole chicken']
Instrucciones: ['Preheat a charcoal grill for medium-high heat, about 375 degrees F (190 degrees C).', 'Mix brown sugar, chili powder, paprika, dry mustard, salt, and black pepper in a small bowl. Place half-full can of beer in the center of a plate.', 'Fit whole chicken over the can of beer with the legs on the bottom; keep upright. Sprinkle 1 teaspoon of seasoning mix into the top cavity of chicken. Beer may foam up when seasonings fall inside the can. Rub remaining seasoning mix over entire surface of chicken.', 'Remove grill plate and push coals to sides of grill. Put grill plate back on and place chicken, standing on the can, over indirect heat. Close the lid and cook chicken until no longer pink at the bone and the juices run clear, about 1 hour 15 minutes. An instant-read thermometer inserted into the thickest part of the thigh, near the bone should read 165 degrees F (74 degrees C).', 'Remove chicken from the grill and let rest upright for 10 minutes before carefully removing the beer can, slicing, and serving; discard beer can.', 'The nutrition data for this recipe includes the full amount of the beer and spice rub ingredients. The actual amount of these ingredients consumed will vary.']

[10] Best Beer Can Chicken | similitud=0.261
URL: https://www.allrecipes.com/recipe/214619/bbq-beer-can-chicken
Descripción: In this best beer can chicken recipe, chicken is seasoned with a sweet and spicy brown sugar-paprika rub and grilled over wood chips for smoky flavor.
Ingredientes: ['2 cups cherry wood chips', '½ cup dark brown sugar', '½ cup salt', '½ cup paprika', '¼ cup ground black pepper', '1 teaspoon cayenne pepper', '2 (12 fluid ounce) cans beer, half full', '2 (3 pound) whole chickens', '¼ cup vegetable oil']
Instrucciones: ['Soak wood chips in water for at least 1 hour.', 'Preheat an outdoor grill for indirect medium heat, about 350 degrees F (175 degrees C).', 'Combine sugar, salt, paprika, black pepper, and cayenne pepper in a small bowl. Place beer cans on a baking sheet. Spoon 1 teaspoon seasoning mix into each can. Be careful, this will make beer foam up and out of the can.', 'Rub each chicken with 2 tablespoons oil. Rub remaining seasoning mix over entire chickens, inside and out. Place each chicken upright over 1 beer can.', "Drain wood chips; place with coals, in an aluminum pan, or under the grill grate according to manufacturer's instructions.", 'Place cans with chicken directly on the grill. Close the lid and grill chicken until no longer pink, and juices run clear, about 1 ½ hours. An instant-read thermometer inserted into the thickest part of thighs should read at least 165 degrees F (74 degrees C).', 'Remove chickens from the grill; discard the beer cans. Cover chickens with a doubled sheet of aluminum foil; rest in a warm area before slicing, 10 minutes.', 'The nutrition data for this recipe includes the full amount of beer, oil, and spice rub ingredients. The actual amount of these ingredients consumed will vary.']


In [60]:
from google import genai


def resolve_gemini_api_key():
    env_names = [
        "api_key"]

    for name in env_names:
        value = os.getenv(name)
        if value:
            return value, name

    if env_path is not None:
        values = dotenv_values(env_path)
        for name in env_names:
            value = values.get(name)
            if value:
                return value, name

    return None, None


def ask_gemini_about_recipes(query, retrieved_results, model_name="gemini-2.5-flash"):
    api_key, api_key_name = resolve_gemini_api_key()
    if not api_key:
        raise RuntimeError(
            f"No se encontró una API key de Gemini en el archivo .env ni en el entorno. Revisado: {', '.join(str(path) for path in env_candidates)}"
        )

    context = build_gemini_context(retrieved_results)
    prompt = f"""
Eres un asistente experto en recetas.
Responde en español, usando SOLO el contexto recuperado del corpus local.
Si el contexto no basta, indícalo explícitamente.

Contexto recuperado:
{context}

Pregunta del usuario:
{query}

Instrucciones:
1. Responde de forma clara y breve.
2. Cita las recetas más relevantes por nombre.
3. Si hay varias opciones, compara sus similitudes y sugiere la mejor.
""".strip()

    client = genai.Client(api_key=api_key)
    response = client.models.generate_content(model=model_name, contents=prompt)
    return response.text, api_key_name


try:
    gemini_response, api_key_name = ask_gemini_about_recipes(query, retrieved_results)
    display(Markdown(gemini_response))
except RuntimeError as exc:
    print(exc)

Aquí tienes varias recetas que utilizan cerveza:

**Bebidas:**
*   **Beer Floats** [1]: Combina cerveza stout de chocolate con helado de café, o cerveza frambuesa ácida con helado de vainilla.
*   **Stout and Ale** [2]: Una bebida conocida como "half and half" en Irlanda, que mezcla cerveza lager con cerveza stout irlandesa.
*   **Irish Slammer** [3]: Un trago que consiste en verter un shot de whiskey y crema irlandesa en un vaso de cerveza stout y beberlo todo de una vez.

**Platos principales con Pollo:**
*   Hay varias recetas de **Beer Can Chicken** (pollo a la cerveza en lata), donde se asa un pollo entero con una lata de cerveza dentro:
    *   **The Best Beer Can Chicken Ever** [5]: Utiliza cerveza stout de chocolate, chiles tailandeses y ajo para un rub de especias complejo.
    *   **Clay's Grilled Beer Can Chicken** [7]: Presenta un rub de especias ligeramente ahumado y usa media lata de cerveza.
    *   **Drunk Chicken** [8]: Un pollo asado a la parrilla con una lata de cerveza en su interior, sazonado con condimento para aves y hojas de laurel.
    *   **Beer Can Chicken** [9]: Un pollo entero sazonado con una mezcla de azúcar moreno, chile en polvo, pimentón y otras especias, asado con media lata de cerveza.
    *   **Best Beer Can Chicken** [10]: Sazona el pollo con un rub dulce y picante de azúcar moreno y pimentón, y se asa a la parrilla con virutas de madera para un sabor ahumado.
*   **Chicken With Stout** [6]: Muslos de pollo fritos y luego cocidos a fuego lento con patatas, zanahorias y col en cerveza stout irlandesa.

**Guarnición:**
*   **Vermont Maple Stout Baked Beans** [4]: Frijoles horneados cocidos a fuego lento con jarabe de arce de Vermont, cerdo salado, cebolla y cerveza stout de arce.

**Comparación y Sugerencia:**
Para **bebidas**, si buscas algo dulce y único, los "Beer Floats" [1] son una excelente opción. Si prefieres bebidas más tradicionales con cerveza, "Stout and Ale" [2] o "Irish Slammer" [3] son las alternativas.

Para **pollo**, si te interesa el método de "Beer Can Chicken", las recetas "The Best Beer Can Chicken Ever" [5] y "Best Beer Can Chicken" [10] parecen ofrecer sabores más complejos, la primera con chocolate stout y chiles, y la segunda con virutas de madera para un toque ahumado. Si buscas un plato de pollo estofado, "Chicken With Stout" [6] es una buena elección.

Finalmente, si quieres un plato de acompañamiento único, "Vermont Maple Stout Baked Beans" [4] ofrece una combinación dulce y salada con cerveza stout.

No se puede determinar "la mejor" receta sin conocer tus preferencias, pero estas son las opciones disponibles y sus particularidades.